In [67]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import sys
import numpy as np

path_pjt = (
    "/Users/Nguye071/Documents/GitHub/Single-and-Multi-level-Fourier-RQMC-for-MSRM"
)
sys.path.insert(0, path_pjt)
sys.path.append(path_pjt + "/2D Gauss Expo Loss")
from analytical_expo_loss import ClosedForm2D
from QMC_MN_expo import RQMC_Fou_MN_expo
from QMC_CV_MN_expo import RQMC_CV_Fou_MN_expo

from MC_loss import MCLossFunction1
from SA_loss import SALoss1
from scipy.optimize import minimize

# 2D case

In [ ]:
maxiter = 3500
mu = [0.0, 0.0]  # Mean vector
sigma = [[1.0, -0.5], [-0.5, 1.0]]  # Covariance matrix
alpha = 1.0  # System-level coupling weight
beta = 1.0  # Risk-aversion coefficient
c = (alpha + 2) / (alpha + 1)  # Inequality-constraint constant
dim = len(mu)
guess = np.zeros(dim)  ## Initial value for optimization procedure
epsilon = 0.8  # Scale used in domain transformation 
#(c < 1, still consistent with the thing mentioned in the paper because we inverse the formula)
N_sobol = 1024  # number of Sobol points
m_shift = 32  # number of randomized digital shifts

## Closed form solution

In [70]:
exactOpt = ClosedForm2D(sigma, beta, alpha)
analytical_sol = exactOpt.compute()

In [71]:
print(f"The analytical solution is {analytical_sol}")

The analytical solution is [0.38689254 0.38689254]


In [ ]:
# =========================
# Print helpers for damping
# =========================
def print_1d_history(vals, title, show=6, ind="damping"):
    print(title)
    print("-" * len(title))
    n = len(vals)
    head = min(show, n)
    for it, d in enumerate(vals[:head]):
        print(f"iter {it:3d} | {ind} = {d:10.4g}")
    if n > show:
        print(f"... ({n - show} more iterations not shown)")


def print_2d_history(vals, title, show=6):
    print(title)
    print("-" * len(title))
    n = len(vals)
    head = min(show, n)
    for it, tup in enumerate(vals[:head]):
        tup_str = "(" + ", ".join(f"{x:10.4g}" for x in tup) + ")"
        print(f"iter {it:3d} | tuple = {tup_str}")
    if n > show:
        print(f"... ({n - show} more iterations not shown)")

## Single-level Fourier-RQMC

In [ ]:
# =========================
# Build single-level loss object
# =========================
loss_qmc_Fou_exp_2D = RQMC_Fou_MN_expo(
    v_mu=mu,
    v_sigma=sigma,
    v_beta=beta,
    N_sobol=N_sobol,
    m_shift=m_shift,
    alpha=alpha,
    c=c,
    epsilon=epsilon,
)


# =========================
# Internal bookkeeping helpers at each iteration
# =========================
def _set_current_m(m):
    """Store current iterate information"""
    m = np.asarray(m, dtype=float)
    loss_qmc_Fou_exp_2D._current_m = m

    if loss_qmc_Fou_exp_2D._pending_jac is not None:
        loss_qmc_Fou_exp_2D._jac_components.append(loss_qmc_Fou_exp_2D._pending_jac)
        loss_qmc_Fou_exp_2D._pending_jac = None

    loss_qmc_Fou_exp_2D.record_qmc_stats()


## Objective: \sum_{k=1}^{d} m_k
def objective_wrapper(m):
    m = np.asarray(m, dtype=float)
    loss_qmc_Fou_exp_2D._current_m = m.copy()
    return loss_qmc_Fou_exp_2D.objective(m)


## Gradient of the objective
def jac_wrapper(m):
    m = np.asarray(m, dtype=float)
    loss_qmc_Fou_exp_2D._current_m = m.copy()
    return loss_qmc_Fou_exp_2D.objective_jac(m)


def scipy_callback(m):
    _set_current_m(m)


# =========================
# Warm-up evaluation at initial point of the optimization
# =========================
loss_qmc_Fou_exp_2D._current_m = np.asarray(guess, dtype=float)
loss_qmc_Fou_exp_2D.shortfall_risk(guess)
loss_qmc_Fou_exp_2D.shortfall_risk_jac(guess)
_set_current_m(guess)

# =========================
# Constraint E[l(X - m)] and its gradient E[∇l(X - m)]
# =========================
cons_qmc_exact = {
    "type": "ineq",
    "fun": lambda x: loss_qmc_Fou_exp_2D.ineq_constraint(x),
    "jac": lambda x: loss_qmc_Fou_exp_2D.ineq_constraint_jac(x),
}

# =========================
# Optimization
# =========================
res = minimize(
    fun=objective_wrapper,
    x0=guess,
    jac=jac_wrapper,
    constraints=cons_qmc_exact,
    method="SLSQP",
    options={"maxiter": maxiter, "ftol": 1e-6},
    callback=scipy_callback,
)


# =========================
# Damping histories
# =========================
print(f"Number of iterations: {res.nit}")

vals_1d = loss_qmc_Fou_exp_2D._history_1d[(0, True)]["K"]
print_1d_history(
    vals_1d,
    "First 1D Fourier component damping K (branch K > beta) by iteration",
    show=6,
)

vals_2d = loss_qmc_Fou_exp_2D._history_2d[(0, 1, True, True)]["K"]
print_2d_history(
    vals_2d,
    "First 2D Fourier component damping K (branch K > beta) by iteration",
    show=6,
)

# =========================
# Final solution + relative error
# =========================
max_qmc_err = loss_qmc_Fou_exp_2D.statistical_error_sol_RQMC(res.x)
rel_err = max_qmc_err / np.max(res.x)

print("\nFinal result")
print("------------")
print(f"Single Fourier-RQMC solution: {res.x}")
print(f"Relative statistical error : {rel_err:.6g}")

Number of iterations: 5
First 1D Fourier component damping K (branch K > beta) by iteration
-------------------------------------------------------------------
iter   0 | damping =      1.618
iter   1 | damping =      1.708
iter   2 | damping =      1.737
iter   3 | damping =      1.739
iter   4 | damping =      1.739
iter   5 | damping =      1.739
First 2D Fourier component damping K (branch K > beta) by iteration
-------------------------------------------------------------------
iter   0 | tuple = (         2,          2)
iter   1 | tuple = (     2.224,      2.224)
iter   2 | tuple = (     2.301,      2.301)
iter   3 | tuple = (     2.306,      2.306)
iter   4 | tuple = (     2.306,      2.306)
iter   5 | tuple = (     2.306,      2.306)

Final result
------------
Single Fourier-RQMC solution: [0.38687942 0.38687942]
Relative statistical error : 0.00125587


## Multilevel Fourier-RQMC

In [ ]:
# =========================
# Build multilevel Fourier-RQMC loss object
# =========================
loss_qmc_CV_Fou_exp_2D = RQMC_CV_Fou_MN_expo(
    v_mu=mu,
    v_sigma=sigma,
    v_beta=beta,
    N_sobol=N_sobol,
    m_shift=m_shift,
    alpha=alpha,
    c=c,
    epsilon=epsilon,
)

# =========================
# Internal bookkeeping helpers at each iteration
# =========================
#
N_sobol_iter = []  # List to keep track for number of Sobol points
count = 0  # This will use to keep track when we enter the local convergence region


def _set_current_m(m):
    """Store current iterate."""
    global count
    m = np.asarray(m, dtype=float)
    loss_qmc_CV_Fou_exp_2D._current_m = m

    # The commit option in this case help us to store only succesful optimization iteration
    loss_qmc_CV_Fou_exp_2D.shortfall_risk(m, commit=True)
    loss_qmc_CV_Fou_exp_2D.shortfall_risk_jac(m, commit=True)
    if loss_qmc_CV_Fou_exp_2D._pending_jac is not None:
        loss_qmc_CV_Fou_exp_2D._jac_components.append(
            loss_qmc_CV_Fou_exp_2D._pending_jac
        )
    loss_qmc_CV_Fou_exp_2D._pending_jac = None
    count += 1

    ## Adaptive choice of Sobol point when entering local convergence region
    N_sobol_iter.append(loss_qmc_CV_Fou_exp_2D._N_current)
    if count > 2:
        loss_qmc_CV_Fou_exp_2D._divide_sobol(mult_factor=1 / 2)

    loss_qmc_CV_Fou_exp_2D.record_qmc_stats()


## Objective: \sum_{k=1}^{d} m_k
def objective_wrapper(m):
    m = np.asarray(m, dtype=float)
    loss_qmc_CV_Fou_exp_2D._current_m = m.copy()
    return loss_qmc_CV_Fou_exp_2D.objective(m)


## Gradient of the objective
def jac_wrapper(m):
    m = np.asarray(m, dtype=float)
    loss_qmc_CV_Fou_exp_2D._current_m = m.copy()
    return loss_qmc_CV_Fou_exp_2D.objective_jac(m)


def scipy_callback(m):
    _set_current_m(m)


# =========================
# Warm-up evaluation at initial point of the optimization
# =========================
loss_qmc_CV_Fou_exp_2D._current_m = np.asarray(guess, dtype=float)
loss_qmc_CV_Fou_exp_2D.shortfall_risk(guess)
loss_qmc_CV_Fou_exp_2D.shortfall_risk_jac(guess)
_set_current_m(guess)

# =========================
# Constraint E[l(X - m)] and its gradient E[∇l(X - m)]
# =========================
cons_qmc_exact = {
    "type": "ineq",
    "fun": lambda x: loss_qmc_CV_Fou_exp_2D.ineq_constraint(x),
    "jac": lambda x: loss_qmc_CV_Fou_exp_2D.ineq_constraint_jac(x),
}

# =========================
# Optimization
# =========================
res = minimize(
    fun=objective_wrapper,
    x0=guess,
    jac=jac_wrapper,
    constraints=cons_qmc_exact,
    method="SLSQP",
    options={"maxiter": maxiter, "ftol": 1e-6},
    callback=scipy_callback,
)


# =========================
# Damping and Sobol point histories
# =========================
print(f"Number of iterations: {res.nit}")

vals_1d = loss_qmc_CV_Fou_exp_2D._history_1d[(0, True)]["K"]
print_1d_history(
    vals_1d,
    "First 1D Fourier component damping K (branch K > beta) by iteration",
    show=6,
)

vals_2d = loss_qmc_CV_Fou_exp_2D._history_2d[(0, 1, True, True)]["K"]
print_2d_history(
    vals_2d,
    "First 2D Fourier component damping K (branch K > beta) by iteration",
    show=6,
)

print_1d_history(N_sobol_iter, "Number of Sobol points", show=6, ind="N_Sobol")
# =========================
# Final solution + relative error
# =========================
max_qmc_err = loss_qmc_CV_Fou_exp_2D.statistical_error_sol_RQMC(res.x)
rel_err = max_qmc_err / np.max(res.x)

print("\nFinal result")
print("------------")
print(f"Multilevel Fourier-RQMC solution: {res.x}")
print(f"Relative statistical error : {rel_err:.6g}")

Number of iterations: 5
First 1D Fourier component damping K (branch K > beta) by iteration
-------------------------------------------------------------------
iter   0 | damping =      1.618
iter   1 | damping =      1.708
iter   2 | damping =      1.642
iter   3 | damping =       1.62
iter   4 | damping =      1.618
iter   5 | damping =      1.618
First 2D Fourier component damping K (branch K > beta) by iteration
-------------------------------------------------------------------
iter   0 | tuple = (         2,          2)
iter   1 | tuple = (     2.224,      2.224)
iter   2 | tuple = (      2.06,       2.06)
iter   3 | tuple = (     2.004,      2.004)
iter   4 | tuple = (         2,          2)
iter   5 | tuple = (         2,          2)
Number of Sobol points
----------------------
iter   0 | N_Sobol =       1024
iter   1 | N_Sobol =       1024
iter   2 | N_Sobol =       1024
iter   3 | N_Sobol =        512
iter   4 | N_Sobol =        256
iter   5 | N_Sobol =        128

Final res

## SAA

In [ ]:
import math
import scipy.stats

MC_points = 10**5  ## Number of MC points
rv = scipy.stats.multivariate_normal(mean=mu, cov=sigma, allow_singular=True)
X = rv.rvs(size=MC_points)

# =========================
# Build SAA loss object
# =========================
loss_MC_2D = MCLossFunction1(X, alpha=alpha, c=c)


def _set_current_m(m):
    """Store current iterate."""
    m = np.asarray(m, dtype=float)
    loss_MC_2D._current_m = m
    loss_MC_2D._var_iter.append(loss_MC_2D._pending_var)
    loss_MC_2D._pending_var = None


# =========================
# Internal bookkeeping helpers at each iteration
# =========================

## Objective: \sum_{k=1}^{d} m_k


def objective_wrapper(m):
    m = np.asarray(m, dtype=float)
    loss_MC_2D._current_m = m.copy()
    return loss_MC_2D.objective(m)


## Gradient of the objective
def jac_wrapper(m):
    m = np.asarray(m, dtype=float)
    loss_MC_2D._current_m = m.copy()
    return loss_MC_2D.objective_jac(m)


def scipy_callback(m):
    _set_current_m(m)


# =========================
# Warm-up evaluation at initial point of the optimization
# =========================
loss_MC_2D._current_m = guess
loss_MC_2D.shortfall_risk(guess)
loss_MC_2D.shortfall_risk_jac(guess)
loss_MC_2D._var_iter.append(loss_MC_2D._pending_var)

# =========================
# Constraint E[l(X - m)] and its gradient E[∇l(X - m)]
# =========================
cons = {
    "type": "ineq",
    "fun": lambda x: loss_MC_2D.ineq_constraint(x),
    "jac": lambda x: loss_MC_2D.ineq_constraint_jac(x),
}

# =========================
# Optimization
# =========================
res = minimize(
    fun=objective_wrapper,
    x0=guess,
    jac=jac_wrapper,
    constraints=cons,
    method="SLSQP",
    options={"maxiter": maxiter, "ftol": 1e-6},
    callback=scipy_callback,
)

# =========================
# Final solution + relative error
# =========================
print(f"Number of iterations: {res.nit}")
max_mc_err = loss_MC_2D.statistical_error_sol_MC(res.x)
rel_err = max_mc_err / np.max(res.x)

print("\nFinal result")
print("------------")

print(f"SAA solution: {res.x}")
print(f"Relative statistical error : {rel_err:.6g}")

Number of iterations: 5

Final result
------------
SAA solution: [0.38790993 0.37394996]
Relative statistical error : 0.0192586


## SA

In [76]:
## Paramters for SA set up based on https://github.com/AchrafTamtalini/SRM/tree/main/Simulation%20MSRM/MC
c_SA, t_SA, gamma_SA = 2, 10, 0.7
epsilon_jac = 10 ** (-6)
K_SA = [[0, 2], [0, 2], [0, 2]]
guess_SA = [0, 0, 0]

In [ ]:
rv = scipy.stats.multivariate_normal(mean=mu, cov=sigma, allow_singular=True)
X = rv.rvs(size=MC_points)
# =========================
# Build SA loss object
# =========================
loss_SA_2D = SALoss1(X, c_SA, gamma_SA, K_SA, t_SA, guess_SA, epsilon_jac, beta, alpha)
Z1 = loss_SA_2D.setRM()
loss_SA_2D.setEst()
# =========================
# Final solution + relative error
# =========================
sa_sol = Z1[:, -1][:2]
print(f"SA solution: {sa_sol}")
_, max_sa_err, _ = loss_SA_2D.getPR()
rel_err = max_sa_err / np.max(sa_sol)
print(f"Relative statistical error : {rel_err:.6g}")

SA solution: [0.37548549 0.38996004]
Relative statistical error : 0.0245577
